In [23]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import RFE,chi2,SelectKBest,f_classif
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from scipy.stats import chi2_contingency
from sklearn.metrics import mean_squared_error,r2_score

In [11]:
data = pd.read_csv('student_chart.csv')

In [12]:
X = data[['age','Gender_label','education_label','internet_label',
          'avg_screen_time','total_study_hours','sleep_hours','mental_health_score',
         'caffeine_intake_mg', 'exercise_minutes','part_time_job','focus_index',
          'burnout_level', 'productivity_score',]]

y = data[ 'exam_score']

In [16]:
def model_prediction(X,name):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    LR = LinearRegression()
    LR.fit(X_train,y_train)
    
    y_pred = LR.predict(X_test)
    print(f' Feature Selection : {name}')
    print("MSE:", mean_squared_error(y_test, y_pred))
    print("R2 Score:", r2_score(y_test, y_pred))


In [20]:
model_prediction(X,'before')

 Feature Selection : before
MSE: 24.99708472825876
R2 Score: 0.8165486357289033


In [19]:
filter_method = SelectKBest(score_func=f_classif, k=10)
X_filter = filter_method.fit_transform(X_train, y_train)

print("Filter Selected Features:")
print(X_train.columns[filter_method.get_support()])
model_prediction(X[X_train.columns[filter_method.get_support()]],'filter method(select k best)')

Filter Selected Features:
Index(['age', 'education_label', 'total_study_hours', 'sleep_hours',
       'mental_health_score', 'caffeine_intake_mg', 'part_time_job',
       'focus_index', 'burnout_level', 'productivity_score'],
      dtype='object')
 Feature Selection : filter method(select k best)
MSE: 24.81032720485705
R2 Score: 0.8179192324536168


In [21]:
model = LinearRegression()
rfe = RFE(model, n_features_to_select=10)
X_wrapper = rfe.fit_transform(X_train, y_train)

print("\nWrapper Selected Features:")
print(X_train.columns[rfe.support_])
model_prediction(X[X_train.columns[rfe.support_]],'wrapper method(rfe)')


Wrapper Selected Features:
Index(['Gender_label', 'education_label', 'internet_label', 'avg_screen_time',
       'total_study_hours', 'sleep_hours', 'part_time_job', 'focus_index',
       'burnout_level', 'productivity_score'],
      dtype='object')
 Feature Selection : wrapper method(rfe)
MSE: 24.979669545435613
R2 Score: 0.8166764441946772


In [24]:
rf = RandomForestRegressor()
rf.fit(X_train, y_train)

importances = rf.feature_importances_
indices = np.argsort(importances)[-10:]

print("\nEmbedded Selected Features:")
print(X_train.columns[indices])
model_prediction(X[X_train.columns[indices]],'embedded method(rf)')


Embedded Selected Features:
Index(['mental_health_score', 'age', 'total_study_hours', 'sleep_hours',
       'avg_screen_time', 'caffeine_intake_mg', 'exercise_minutes',
       'focus_index', 'burnout_level', 'productivity_score'],
      dtype='object')
 Feature Selection : embedded method(rf)
MSE: 24.82166569933352
R2 Score: 0.8178360202589507


In [29]:
scaler_standard = StandardScaler()
X_train_standard = scaler_standard.fit_transform(X_train)
model_prediction(X_train_standard,'scalar standard')
# Min-Max Scaling
scaler_minmax = MinMaxScaler()
X_train_minmax = scaler_minmax.fit_transform(X_train)

print("\nFeature Scaling Completed Successfully")
#model_prediction(X_train_minmax.columns[rfe.support_],'min-max')

ValueError: Found input variables with inconsistent numbers of samples: [4000, 5000]